# Database Integration

## Objective

Establish a connection between Python and PostgreSQL, prepare the project database, and load the validated analytical datasets into PostgreSQL.

In [9]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())
print("Source directory exists:", (PROJECT_ROOT / "src").exists())

Project root: C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization
Project root exists: True
Source directory exists: True


In [1]:
import sqlalchemy
import psycopg

print("SQLAlchemy:", sqlalchemy.__version__)
print("Psycopg:", psycopg.__version__)

SQLAlchemy: 2.0.48
Psycopg: 3.3.2


In [2]:
from sqlalchemy import create_engine, text
from getpass import getpass

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "postgres"
DB_USER = "postgres"



DB_PASSWORD = getpass("Enter PostgreSQL password: ")

DATABASE_URL = (
    f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

with engine.connect() as connection:
    result = connection.execute(text("SELECT version()"))
    print(result.scalar())

PostgreSQL 17.7 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit


## 1. Database Schema Design

Inspect the validated analytical dataset and its column structure before designing the PostgreSQL tables.

The cleaned Parquet dataset serves as the reference for the primary analytical table.

In [3]:
import pandas as pd
from pathlib import Path

CLEAN_DATA_PATH = Path("../data/processed/online_retail_clean.parquet")

print("Dataset exists:", CLEAN_DATA_PATH.exists())

df_schema = pd.read_parquet(CLEAN_DATA_PATH)

print("\nDataset dimensions:", df_schema.shape)

print("\nColumn names and data types:")
print(df_schema.dtypes.to_string())

print("\nSample records:")
display(df_schema.head(5))

Dataset exists: True

Dataset dimensions: (1033036, 33)

Column names and data types:
Invoice                                   string
StockCode                                 string
Description                               string
Quantity                                   int64
InvoiceDate                       datetime64[us]
Price                                    float64
Customer ID                              float64
Country                                   string
YearMonth                              period[M]
Description_was_missing                     bool
is_cancellation                             bool
has_customer_id                             bool
has_positive_quantity                       bool
has_positive_price                          bool
has_positive_line_value                     bool
candidate_paid_purchase                     bool
Description_normalized                    string
is_zero_price                               bool
zero_price_category             

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth,Description_was_missing,...,customer_level_eligible,qualifying_purchase,notes,is_merchandise_revenue_record,is_customer_analytics_eligible,is_qualifying_purchase,merchandise_revenue,qualifying_purchase_revenue,is_excess_exact_duplicate,original_row_position
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12,False,...,True,True,Identified-customer merchandise transactions,True,True,True,83.4,83.4,False,0
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12,False,...,True,True,Identified-customer merchandise transactions,True,True,True,81.0,81.0,False,1
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12,False,...,True,True,Identified-customer merchandise transactions,True,True,True,81.0,81.0,False,2
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12,False,...,True,True,Identified-customer merchandise transactions,True,True,True,100.8,100.8,False,3
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12,False,...,True,True,Identified-customer merchandise transactions,True,True,True,30.0,30.0,False,4


In [4]:
# 1. Customer ID integrity
customer_ids = df_schema["Customer ID"].dropna()

print("CUSTOMER ID CHECK")
print("Missing values:", df_schema["Customer ID"].isna().sum())
print("Non-integer values:", (customer_ids % 1 != 0).sum())
print("Minimum:", customer_ids.min())
print("Maximum:", customer_ids.max())

# 2. Original row position integrity
print("\nORIGINAL ROW POSITION CHECK")
print("Missing values:", df_schema["original_row_position"].isna().sum())
print("Unique values:", df_schema["original_row_position"].nunique())
print("Total records:", len(df_schema))
print("Suitable as primary key:",
      df_schema["original_row_position"].is_unique
      and df_schema["original_row_position"].notna().all())

# 3. Reference table structures
reference_dir = Path("../data/reference")

description_mapping = pd.read_csv(
    reference_dir / "description_mapping.csv"
)

transaction_taxonomy = pd.read_csv(
    reference_dir / "transaction_taxonomy.csv"
)

print("\nDESCRIPTION MAPPING")
print("Dimensions:", description_mapping.shape)
print(description_mapping.dtypes)
display(description_mapping.head())

print("\nTRANSACTION TAXONOMY")
print("Dimensions:", transaction_taxonomy.shape)
print(transaction_taxonomy.dtypes)
display(transaction_taxonomy.head())

CUSTOMER ID CHECK
Missing values: 235151
Non-integer values: 0
Minimum: 12346.0
Maximum: 18287.0

ORIGINAL ROW POSITION CHECK
Missing values: 0
Unique values: 1033036
Total records: 1033036
Suitable as primary key: True

DESCRIPTION MAPPING
Dimensions: (1387, 3)
StockCode                str
imputed_description      str
evidence_count         int64
dtype: object


,StockCode,imputed_description,evidence_count
0,79321,CHILLI LIGHTS,1211
1,22084,PAPER CHAIN KIT EMPIRE,843
2,37461,FUNKY MONKEY MUG,98
3,37509,NEW ENGLAND MUG W GIFT BOX,18
4,84977,WIRE FLOWER T-LIGHT HOLDER,45



TRANSACTION TAXONOMY
Dimensions: (16, 5)
transaction_category              str
merchandise_revenue_treatment     str
customer_level_eligible          bool
qualifying_purchase              bool
notes                             str
dtype: object


,transaction_category,merchandise_revenue_treatment,customer_level_eligible,qualifying_purchase,notes
0,merchandise_candidate,include_positive,True,True,Identified-customer merchandise transactions
1,anonymous_merchandise_candidate,include_positive,False,False,Merchandise transactions without Customer ID
2,merchandise_cancellation_candidate,include_negative,True,False,Identified-customer merchandise cancellations
3,anonymous_merchandise_cancellation_candidate,include_negative,False,False,Anonymous merchandise cancellations
4,delivery_charge,separate,False,False,Delivery-related charges


In [5]:
reference_checks = {
    "Description mapping StockCode": (
        description_mapping,
        "StockCode"
    ),
    "Transaction taxonomy category": (
        transaction_taxonomy,
        "transaction_category"
    )
}

for name, (reference_df, key_column) in reference_checks.items():
    print(f"\n{name}")
    print("Total records:", len(reference_df))
    print("Unique keys:", reference_df[key_column].nunique())
    print("Missing keys:", reference_df[key_column].isna().sum())
    print(
        "Primary key eligible:",
        reference_df[key_column].is_unique
        and reference_df[key_column].notna().all()
    )


Description mapping StockCode
Total records: 1387
Unique keys: 1387
Missing keys: 0
Primary key eligible: True

Transaction taxonomy category
Total records: 16
Unique keys: 16
Missing keys: 0
Primary key eligible: True


In [6]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

project_database_url = URL.create(
    drivername="postgresql+psycopg",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database="customer_intelligence"
)

project_engine = create_engine(project_database_url)

with project_engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), version()")
    )

    database_name, postgres_version = result.fetchone()

    print("Connected database:", database_name)
    print("PostgreSQL version:", postgres_version)

Connected database: customer_intelligence
PostgreSQL version: PostgreSQL 17.7 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit


In [7]:
from sqlalchemy import inspect

# Map source column names to PostgreSQL column names.
column_mapping = {
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country",
    "YearMonth": "year_month",
    "Description_was_missing": "description_was_missing",
    "Description_normalized": "description_normalized",
    "StockCode_normalized": "stock_code_normalized",
}

# Apply standard lowercase snake_case to remaining columns.
source_columns = [
    column_mapping.get(
        column,
        column.lower().replace(" ", "_")
    )
    for column in df_schema.columns
]

inspector = inspect(project_engine)

for table_name in [
    "retail_transactions",
    "duplicate_transaction_audit"
]:
    database_columns = [
        column["name"]
        for column in inspector.get_columns(
            table_name,
            schema="public"
        )
    ]

    missing_columns = set(source_columns) - set(database_columns)
    unexpected_columns = set(database_columns) - set(source_columns)

    print(f"\n{table_name}")
    print("Database columns:", len(database_columns))
    print("Expected columns:", len(source_columns))
    print("Missing columns:", missing_columns)
    print("Unexpected columns:", unexpected_columns)
    print("Schema compatible:", not missing_columns and not unexpected_columns)


retail_transactions
Database columns: 33
Expected columns: 33
Missing columns: set()
Unexpected columns: set()
Schema compatible: True

duplicate_transaction_audit
Database columns: 33
Expected columns: 33
Missing columns: set()
Unexpected columns: set()
Schema compatible: True
